In [ ]:
# 1) Création du venv
!python -m venv .venv

# 2) Installation des libs dans le venv via le pip du venv
!.venv\Scripts\python -m pip install --upgrade pip
!.venv\Scripts\pip install -r requirements.txt

# 🏢 Projet ML - Système de Prédiction RH

Ce notebook implémente un système complet de comparaison de modèles ML pour prédire l'attrition des employés (départ de l'entreprise).

## 📋 Contenu
1. **Prétraitement des données** - Extraction et fusion des fichiers CSV
2. **Chargement et préparation des données**
3. **Définition des modèles**
4. **Réglage des hyperparamètres (optionnel)**
5. **Entraînement et évaluation**
6. **Comparaison des modèles**
7. **Validation croisée**

## 🔧 ÉTAPE 0 - Prétraitement des Données Brutes

### Extraction du fichier ZIP

Les données sont dans un fichier `in_out_time.zip` qui contient plusieurs fichiers CSV.

In [ ]:
import shutil
import os

data_root="data/"

# Extraire le fichier ZIP s'il existe
if os.path.exists(data_root + 'in_out_time.zip'):
    print("📦 Extraction du fichier in_out_time.zip...")
    shutil.unpack_archive(data_root + 'in_out_time.zip', data_root)
    print("✅ Extraction terminée")
else:
    print("⚠️ Le fichier in_out_time.zip n'existe pas. Vérifiez que les fichiers CSV sont déjà extraits.")

### Fusion des fichiers CSV

Les données RH sont réparties dans 5 fichiers différents :
- `general_data.csv` - Données générales des employés
- `employee_survey_data.csv` - Enquête auprès des employés
- `manager_survey_data.csv` - Enquête auprès des managers
- `in_time.csv` - Heures d'arrivée
- `out_time.csv` - Heures de départ

Nous allons les fusionner via l'`EmployeeID`.

In [ ]:
import pandas as pd
import glob

# Vérifier la présence des fichiers
required_files = [data_root + 'general_data.csv', data_root + 'employee_survey_data.csv', data_root + 'manager_survey_data.csv', 
                  data_root + 'in_time.csv', data_root + 'out_time.csv']
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"❌ Fichiers manquants: {missing_files}")
    print("⚠️ Assurez-vous d'avoir extrait le fichier ZIP ou que les CSV sont présents.")
else:
    print("📊 Chargement des fichiers CSV...")
    
    # Charger les fichiers principaux
    df1 = pd.read_csv(data_root + 'general_data.csv')
    df2 = pd.read_csv(data_root + 'employee_survey_data.csv')
    df3 = pd.read_csv(data_root + 'manager_survey_data.csv')
    
    print(f"  - general_data.csv: {df1.shape}")
    print(f"  - employee_survey_data.csv: {df2.shape}")
    print(f"  - manager_survey_data.csv: {df3.shape}")
    
    # Charger les fichiers temporels
    df4 = pd.read_csv(data_root + 'in_time.csv')
    df5 = pd.read_csv(data_root + 'out_time.csv')
    
    print(f"  - in_time.csv: {df4.shape}")
    print(f"  - out_time.csv: {df5.shape}")
    
    # Renommer la première colonne des fichiers in/out time
    df4 = df4.rename(columns={'Unnamed: 0': 'EmployeeID'})
    df5 = df5.rename(columns={'Unnamed: 0': 'EmployeeID'})
    
    print("\n🔧 Création de features agrégées à partir des données temporelles...")
    print("   (Pour éviter l'explosion de mémoire avec 1M+ colonnes)")
    
    # Créer des features agrégées au lieu d'utiliser toutes les colonnes de dates
    def create_time_features(df_time, prefix='in'):
        """Créer des features agrégées à partir des heures d'arrivée/départ"""
        # Sélectionner seulement les colonnes de dates (pas EmployeeID)
        date_cols = [col for col in df_time.columns if col != 'EmployeeID']
        
        # Convertir en format datetime
        df_values = df_time[date_cols].apply(pd.to_datetime, errors='coerce')
        
        # Extraire l'heure en format numérique (heures + minutes/60)
        df_hours = df_values.apply(lambda x: x.dt.hour + x.dt.minute/60.0)
        
        # Calculer des statistiques agrégées
        features = pd.DataFrame()
        features['EmployeeID'] = df_time['EmployeeID']
        features[f'{prefix}_avg_hour'] = df_hours.mean(axis=1)  # Heure moyenne
        features[f'{prefix}_std_hour'] = df_hours.std(axis=1)   # Variabilité
        features[f'{prefix}_min_hour'] = df_hours.min(axis=1)   # Plus tôt
        features[f'{prefix}_max_hour'] = df_hours.max(axis=1)   # Plus tard
        features[f'{prefix}_missing_days'] = df_values.isna().sum(axis=1)  # Jours manquants
        
        return features
    
    # Créer les features pour in_time et out_time
    in_features = create_time_features(df4, prefix='arrival')
    out_features = create_time_features(df5, prefix='departure')
    
    # Fusionner les features temporelles
    time_features = in_features.merge(out_features, on='EmployeeID', how='inner')
    
    # Calculer le temps de travail moyen
    time_features['avg_work_hours'] = (
        time_features['departure_avg_hour'] - time_features['arrival_avg_hour']
    )
    
    print(f"  - Features temporelles créées: {time_features.shape[1]-1} colonnes")
    
    print("\n🔗 Fusion des données...")
    # Fusion en chaîne (2 par 2)
    resultat = df1.merge(df2, on='EmployeeID', how='inner') \
                  .merge(df3, on='EmployeeID', how='inner') \
                  .merge(time_features, on='EmployeeID', how='inner')
    
    # Suppression des colonnes inutiles
    colonnes_a_supprimer = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeID']
    colonnes_existantes = [col for col in colonnes_a_supprimer if col in resultat.columns]
    
    if colonnes_existantes:
        resultat = resultat.drop(colonnes_existantes, axis=1)
        print(f"  - Colonnes supprimées: {colonnes_existantes}")
    
    # Sauvegarder
    resultat.to_csv('Dataset_clean.csv', index=False)
    
    print(f"\n✅ Dataset fusionné et sauvegardé dans 'Dataset_clean.csv'")
    print(f"   Dimensions: {resultat.shape[0]} lignes × {resultat.shape[1]} colonnes")
    print(f"\n📋 Aperçu des colonnes:")
    print(f"   Colonnes générales: {list(df1.columns[:5])}...")
    print(f"   Features temporelles: {list(time_features.columns[1:6])}...")
    
    # Afficher les premières lignes
    print(f"\n📊 Aperçu des données:")
    display(resultat.head())
    
    # Vérifier les valeurs manquantes
    print(f"\n🔍 Valeurs manquantes par colonne:")
    missing = resultat.isnull().sum()
    if missing.sum() > 0:
        display(missing[missing > 0])
    else:
        print("   Aucune valeur manquante !")

## 1️⃣ Chargement et Préparation des Données

Nous allons utiliser le fichier `Dataset_clean.csv` créé lors du prétraitement.

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modèles
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV

# Métriques
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix, 
    roc_curve, precision_recall_curve, average_precision_score
)

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Imports terminés")

## 1️⃣ Imports et Configuration

In [ ]:
def get_data_pipeline(csv_path, target_column='Attrition'):
    """
    Créer le pipeline de chargement et prétraitement des données avec split 70/15/15:
    1. Remplir les valeurs manquantes (médiane pour numérique, mode pour catégoriel)
    2. Transformer les données catégorielles avec OneHotEncoder
    3. Standardisation
    4. Split en 3 ensembles : Train (70%), Validation (15%), Test (15%)
    
    Args:
        csv_path: chemin vers le fichier CSV
        target_column: nom de la colonne cible (par défaut 'Attrition')
    """
    print(f"[Loader] Chargement des données depuis: {csv_path}")
    
    # 1. CHARGER LES DONNÉES
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
    except FileNotFoundError:
        raise FileNotFoundError(f"Fichier introuvable: {csv_path}")

    print(f"[Loader] Dataset chargé: {df.shape[0]} lignes × {df.shape[1]} colonnes")
    
    # Vérifier si la colonne cible existe
    if target_column not in df.columns:
        print(f"\n⚠️ La colonne '{target_column}' n'existe pas.")
        print(f"📋 Colonnes disponibles: {list(df.columns[:20])}")
        raise ValueError(f"Colonne cible '{target_column}' introuvable dans le dataset")
    
    # Séparer features X et target y
    blacklist = []  # Liste pour éviter les fuites de données si nécessaire
    cols_to_drop = [c for c in blacklist if c in df.columns]
    
    if cols_to_drop:
        df = df.drop(cols_to_drop, axis=1)
        print(f"[Loader] Colonnes supprimées: {cols_to_drop}")
    
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    
    # Convertir la cible en format binaire si nécessaire (Yes/No -> 1/0)
    if y.dtype == 'object':
        print(f"[Loader] Conversion de la variable cible en format binaire")
        y = y.map({'Yes': 1, 'No': 0})
        if y.isnull().any():
            print(f"⚠️ Valeurs non converties détectées dans la cible")

    # 2. SPLIT 70/15/15 (train/validation/test)
    print(f"\n[Loader] Split des données: 70% Train / 15% Validation / 15% Test")
    
    # Premier split : 70% train, 30% temp (validation + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    # Deuxième split : diviser les 30% en 15% validation et 15% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # 3. PIPELINE DE PRÉTRAITEMENT
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    print(f"[Loader] Features numériques: {len(numeric_features)}")
    print(f"[Loader] Features catégorielles: {len(categorical_features)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

print("✅ Fonction de chargement définie (split 70/15/15)")

In [ ]:
# Charger les données avec split 70/15/15
print("🔄 Chargement des données RH...\n")

# Vérifier si le fichier existe
if not os.path.exists("Dataset_clean.csv"):
    print("❌ Le fichier 'Dataset_clean.csv' n'existe pas.")
    print("⚠️ Veuillez exécuter les cellules de prétraitement ci-dessus d'abord.")
    raise FileNotFoundError("Dataset_clean.csv introuvable. Exécutez d'abord le prétraitement.")

# Charger avec la colonne cible appropriée (Attrition pour les données RH)
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = get_data_pipeline(
    "Dataset_clean.csv", 
    target_column='Attrition'  # Changez si votre colonne cible a un autre nom
)

print(f"\n📊 Dimensions des données (70/15/15):")
print(f"   - Train:      X={X_train.shape}, y={y_train.shape}")
print(f"   - Validation: X={X_val.shape}, y={y_val.shape}")
print(f"   - Test:       X={X_test.shape}, y={y_test.shape}")

# Afficher la distribution de la target
print(f"\n🎯 Distribution de la variable cible:")
print(f"\n📊 Train set:")
display(pd.DataFrame({
    'Count': y_train.value_counts(),
    'Proportion': y_train.value_counts(normalize=True)
}))

print(f"\n📊 Validation set:")
display(pd.DataFrame({
    'Count': y_val.value_counts(),
    'Proportion': y_val.value_counts(normalize=True)
}))

print(f"\n📊 Test set:")
display(pd.DataFrame({
    'Count': y_test.value_counts(),
    'Proportion': y_test.value_counts(normalize=True)
}))

## 2️⃣.1 Analyse Exploratoire des Données (EDA)

Avant d'entraîner les modèles, explorons les données pour mieux comprendre leur structure et leurs caractéristiques.

### 📊 Vue d'ensemble du dataset

In [ ]:
# Charger le dataset complet pour l'analyse
df_analysis = pd.read_csv("Dataset_clean.csv")

print("=" * 70)
print("📋 INFORMATIONS GÉNÉRALES SUR LE DATASET")
print("=" * 70)
print(f"\n📊 Dimensions: {df_analysis.shape[0]} lignes × {df_analysis.shape[1]} colonnes")
print(f"\n📂 Types de données:")
print(df_analysis.dtypes.value_counts())

print(f"\n🔍 Aperçu des premières lignes:")
display(df_analysis.head(10))

print(f"\n📈 Statistiques descriptives (variables numériques):")
display(df_analysis.describe())

# Vérifier les valeurs manquantes
print(f"\n⚠️ Valeurs manquantes par colonne:")
missing = df_analysis.isnull().sum()
if missing.sum() > 0:
    missing_df = pd.DataFrame({
        'Colonne': missing[missing > 0].index,
        'Nombre': missing[missing > 0].values,
        'Pourcentage': (missing[missing > 0] / len(df_analysis) * 100).round(2)
    })
    display(missing_df)
else:
    print("   ✅ Aucune valeur manquante !")

### 🎯 Analyse de la Variable Cible (Attrition)

In [ ]:
# Analyser la distribution de la variable cible
print("=" * 70)
print("🎯 ANALYSE DE LA VARIABLE CIBLE: ATTRITION")
print("=" * 70)

# Convertir en format binaire si nécessaire
if df_analysis['Attrition'].dtype == 'object':
    attrition_binary = df_analysis['Attrition'].map({'Yes': 1, 'No': 0})
else:
    attrition_binary = df_analysis['Attrition']

# Distribution
print(f"\n📊 Distribution de l'Attrition:")
attrition_counts = df_analysis['Attrition'].value_counts()
attrition_pct = df_analysis['Attrition'].value_counts(normalize=True) * 100

distribution_df = pd.DataFrame({
    'Statut': attrition_counts.index,
    'Nombre': attrition_counts.values,
    'Pourcentage': attrition_pct.values.round(2)
})
display(distribution_df)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique en barres
attrition_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribution de l\'Attrition', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Attrition', fontsize=12)
axes[0].set_ylabel('Nombre d\'employés', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for i, (idx, val) in enumerate(attrition_counts.items()):
    axes[0].text(i, val, f'{val}\n({attrition_pct.iloc[i]:.1f}%)', 
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Graphique en camembert
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(attrition_counts, labels=attrition_counts.index, autopct='%1.1f%%',
           colors=colors, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Proportion de l\'Attrition', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Déséquilibre des classes
ratio = attrition_counts.max() / attrition_counts.min()
print(f"\n⚖️ Ratio de déséquilibre: {ratio:.2f}:1")
if ratio > 3:
    print("   ⚠️ Le dataset est déséquilibré (envisager le rééchantillonnage ou SMOTE)")
else:
    print("   ✅ Le dataset est relativement équilibré")

### 📈 Distribution des Variables Numériques

In [ ]:
# Visualiser les distributions des principales variables numériques
numeric_cols = df_analysis.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Exclure les colonnes non pertinentes
exclude_cols = ['Attrition']
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

# Sélectionner les 12 variables les plus intéressantes
key_numeric_cols = [
    'Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
    'YearsWithCurrManager', 'YearsSinceLastPromotion', 'DistanceFromHome',
    'NumCompaniesWorked', 'PercentSalaryHike', 'TrainingTimesLastYear',
    'arrival_avg_hour', 'avg_work_hours'
]

# Filtrer celles qui existent
key_numeric_cols = [col for col in key_numeric_cols if col in numeric_cols]

print(f"📊 Visualisation de {len(key_numeric_cols)} variables numériques clés\n")

# Créer les histogrammes
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.ravel()

for idx, col in enumerate(key_numeric_cols[:12]):
    if idx < len(axes):
        # Histogramme avec KDE
        df_analysis[col].hist(bins=30, ax=axes[idx], color='skyblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel('')
        axes[idx].set_ylabel('Fréquence', fontsize=9)
        axes[idx].grid(True, alpha=0.3)
        
        # Ajouter des statistiques
        mean_val = df_analysis[col].mean()
        median_val = df_analysis[col].median()
        axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Moyenne: {mean_val:.1f}')
        axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Médiane: {median_val:.1f}')
        axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n✅ Les distributions permettent d'identifier les valeurs aberrantes et la symétrie des données")

### 📊 Distribution des Variables Catégorielles

In [ ]:
# Analyser les variables catégorielles
categorical_cols = df_analysis.select_dtypes(include=['object']).columns.tolist()

# Exclure Attrition de l'analyse
categorical_cols = [col for col in categorical_cols if col != 'Attrition']

print(f"📊 Analyse de {len(categorical_cols)} variables catégorielles\n")

# Sélectionner les plus importantes
key_categorical_cols = [
    'BusinessTravel', 'Department', 'EducationField', 'Gender',
    'JobRole', 'MaritalStatus'
]

# Filtrer celles qui existent
key_categorical_cols = [col for col in key_categorical_cols if col in categorical_cols]

# Créer les graphiques
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(key_categorical_cols[:6]):
    if idx < len(axes):
        # Compter les valeurs
        value_counts = df_analysis[col].value_counts()
        
        # Graphique en barres
        value_counts.plot(kind='bar', ax=axes[idx], color='coral', edgecolor='black')
        axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('')
        axes[idx].set_ylabel('Nombre d\'employés', fontsize=10)
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].grid(True, alpha=0.3, axis='y')
        
        # Ajouter les valeurs sur les barres
        for i, v in enumerate(value_counts.values):
            axes[idx].text(i, v, f'{v}\n({v/len(df_analysis)*100:.1f}%)', 
                          ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Afficher un résumé tabulaire
print("\n📋 Résumé des variables catégorielles:")
for col in key_categorical_cols:
    print(f"\n{col}:")
    print(df_analysis[col].value_counts())
    print(f"Nombre de catégories uniques: {df_analysis[col].nunique()}")

### 🔗 Analyse des Corrélations

In [ ]:
# Analyser les corrélations entre variables numériques
print("=" * 70)
print("🔗 MATRICE DE CORRÉLATION")
print("=" * 70)

# Convertir Attrition en binaire pour l'analyse de corrélation
df_corr = df_analysis.copy()
if df_corr['Attrition'].dtype == 'object':
    df_corr['Attrition'] = df_corr['Attrition'].map({'Yes': 1, 'No': 0})

# Sélectionner uniquement les colonnes numériques
numeric_df = df_corr.select_dtypes(include=['int64', 'float64'])

# Calculer la matrice de corrélation
correlation_matrix = numeric_df.corr()

# Visualisation de la matrice de corrélation complète
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
           linewidths=0.5, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de Corrélation - Toutes les Variables Numériques', 
         fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Trouver les corrélations les plus fortes avec Attrition
print("\n🎯 Variables les plus corrélées avec Attrition:\n")
attrition_corr = correlation_matrix['Attrition'].drop('Attrition').sort_values(ascending=False)

# Top 15 corrélations positives et négatives
top_positive = attrition_corr.head(10)
top_negative = attrition_corr.tail(10)

# Affichage
corr_df = pd.DataFrame({
    'Variable': list(top_positive.index) + list(top_negative.index),
    'Corrélation': list(top_positive.values) + list(top_negative.values)
})
corr_df = corr_df.sort_values('Corrélation', ascending=False)
display(corr_df)

# Visualisation des top corrélations
fig, ax = plt.subplots(figsize=(12, 8))
top_15 = attrition_corr.abs().sort_values(ascending=True).tail(15)
colors = ['red' if attrition_corr[var] < 0 else 'green' for var in top_15.index]

top_15_values = [attrition_corr[var] for var in top_15.index]
ax.barh(range(len(top_15)), top_15_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15.index, fontsize=10)
ax.set_xlabel('Corrélation avec Attrition', fontsize=12)
ax.set_title('Top 15 des Variables Corrélées avec l\'Attrition', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

# Ajouter les valeurs sur les barres
for i, v in enumerate(top_15_values):
    ax.text(v, i, f' {v:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Les corrélations positives augmentent le risque de départ")
print("💡 Les corrélations négatives diminuent le risque de départ")

### 📊 Analyse Bivariée: Attrition vs Variables Clés

In [ ]:
# Analyser la relation entre Attrition et les variables clés
print("=" * 70)
print("📊 ANALYSE BIVARIÉE: ATTRITION VS VARIABLES CLÉS")
print("=" * 70)

# Variables numériques à analyser
key_vars = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'DistanceFromHome']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, var in enumerate(key_vars):
    if var in df_analysis.columns:
        # Box plot pour comparer les distributions
        df_analysis.boxplot(column=var, by='Attrition', ax=axes[idx])
        axes[idx].set_title(f'{var} vs Attrition', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Attrition', fontsize=11)
        axes[idx].set_ylabel(var, fontsize=11)
        axes[idx].grid(True, alpha=0.3)
        
        # Calculer et afficher les statistiques
        stats = df_analysis.groupby('Attrition')[var].agg(['mean', 'median'])
        textstr = f"Moyenne:\nNo: {stats.loc['No', 'mean']:.1f}\nYes: {stats.loc['Yes', 'mean']:.1f}"
        axes[idx].text(0.02, 0.98, textstr, transform=axes[idx].transAxes,
                      fontsize=9, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('')  # Supprimer le titre automatique
plt.tight_layout()
plt.show()

# Analyse pour variables catégorielles
print("\n📊 Taux d'Attrition par catégorie:\n")

categorical_vars = ['Department', 'JobRole', 'MaritalStatus', 'BusinessTravel']

for var in categorical_vars:
    if var in df_analysis.columns:
        print(f"\n{var}:")
        attrition_rate = df_analysis.groupby(var)['Attrition'].apply(
            lambda x: (x == 'Yes').sum() / len(x) * 100
        ).sort_values(ascending=False)
        
        attrition_df = pd.DataFrame({
            'Catégorie': attrition_rate.index,
            'Taux d\'Attrition (%)': attrition_rate.values.round(2)
        })
        display(attrition_df)

# Visualisation du taux d'attrition par département
if 'Department' in df_analysis.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    dept_attrition = df_analysis.groupby('Department')['Attrition'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).sort_values(ascending=True)
    
    colors_dept = ['#e74c3c' if x > 20 else '#f39c12' if x > 15 else '#2ecc71' 
                   for x in dept_attrition.values]
    
    dept_attrition.plot(kind='barh', ax=ax, color=colors_dept, edgecolor='black')
    ax.set_xlabel('Taux d\'Attrition (%)', fontsize=12)
    ax.set_ylabel('Département', fontsize=12)
    ax.set_title('Taux d\'Attrition par Département', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Ajouter les valeurs
    for i, v in enumerate(dept_attrition.values):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

print("\n✅ L'analyse bivariée révèle les facteurs clés influençant l'attrition")

### 🎨 Insights Clés de l'Analyse Exploratoire

Cette section résume les découvertes principales qui guideront la modélisation.

In [ ]:
# Résumé des insights clés
print("=" * 70)
print("🎨 INSIGHTS CLÉS DE L'ANALYSE EXPLORATOIRE")
print("=" * 70)
print()

# 1. Distribution de la cible
attrition_pct = (df_analysis['Attrition'] == 'Yes').sum() / len(df_analysis) * 100
print(f"📊 1. Variable cible (Attrition):")
print(f"   • Taux d'attrition global: {attrition_pct:.1f}%")
print(f"   • Classe majoritaire: {'No' if attrition_pct < 50 else 'Yes'}")
print()

# 2. Variables numériques importantes
if 'Attrition' in correlation_matrix.columns:
    top_corr = correlation_matrix['Attrition'].abs().sort_values(ascending=False)[1:6]
    print(f"🔗 2. Top 5 variables numériques corrélées:")
    for var, corr in top_corr.items():
        direction = "augmente" if correlation_matrix['Attrition'][var] > 0 else "diminue"
        print(f"   • {var}: {abs(corr):.3f} ({direction} le risque)")
print()

# 3. Variables catégorielles à risque
print(f"📈 3. Catégories à risque élevé d'attrition:")
for var in ['Department', 'JobRole', 'BusinessTravel']:
    if var in df_analysis.columns:
        attrition_by_cat = df_analysis.groupby(var)['Attrition'].apply(
            lambda x: (x == 'Yes').sum() / len(x) * 100
        ).sort_values(ascending=False)
        
        if len(attrition_by_cat) > 0:
            highest = attrition_by_cat.index[0]
            highest_rate = attrition_by_cat.values[0]
            print(f"   • {var}: '{highest}' ({highest_rate:.1f}%)")
print()

# 4. Recommandations pour la modélisation
print(f"💡 4. Recommandations pour la modélisation:")
print(f"   • Stratification nécessaire (classes déséquilibrées)")
print(f"   • Features engineering: interactions possibles entre variables")
print(f"   • Normalisation importante (échelles différentes)")
print(f"   • Encodage des variables catégorielles requis")
print()

# 5. Qualité des données
missing_total = df_analysis.isnull().sum().sum()
print(f"✅ 5. Qualité des données:")
print(f"   • Valeurs manquantes: {missing_total} ({missing_total/df_analysis.size*100:.2f}%)")
print(f"   • Dataset complet: {len(df_analysis)} employés")
print(f"   • Features disponibles: {len(df_analysis.columns)} colonnes")
print()

print("=" * 70)
print("✅ Analyse exploratoire terminée - Prêt pour la modélisation")
print("=" * 70)

## 3️⃣ Définition des Modèles

In [ ]:
def create_logistic_model(preprocessor):
    """Régression Logistique"""
    model = LogisticRegression(random_state=42, max_iter=1000)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_perceptron_model(preprocessor):
    """Perceptron avec calibration pour predict_proba"""
    base_model = Perceptron(random_state=42, max_iter=1000)
    calibrated_model = CalibratedClassifierCV(base_model, cv=3)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', calibrated_model)
    ])
    return pipeline

def create_decision_tree_model(preprocessor):
    """Arbre de Décision"""
    model = DecisionTreeClassifier(random_state=42)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_random_forest_model(preprocessor):
    """Forêt Aléatoire"""
    model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=4)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_svm_model(preprocessor):
    """SVM Linéaire avec calibration"""
    base_model = LinearSVC(random_state=42, max_iter=1000)
    calibrated_model = CalibratedClassifierCV(base_model, cv=3)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', calibrated_model)
    ])
    return pipeline

def create_naive_bayes_model(preprocessor):
    """Naive Bayes (Gaussian)"""
    # Créer un preprocessor modifié pour Naive Bayes (sparse_output=False)
    numeric_features = preprocessor.transformers[0][2]
    categorical_features = preprocessor.transformers[1][2]
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor_nb = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    model = GaussianNB()
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_nb),
        ('classifier', model)
    ])
    return pipeline

print("✅ Fonctions de création de modèles définies")

In [ ]:
# Créer tous les modèles
print("🔨 Création des modèles...\n")

competitors = {
    "Logistic Regression": create_logistic_model(preprocessor),
    "Perceptron": create_perceptron_model(preprocessor),
    "Decision Tree": create_decision_tree_model(preprocessor),
    "Random Forest": create_random_forest_model(preprocessor),
    "Support Vector Machine": create_svm_model(preprocessor),
    "Naive Bayes": create_naive_bayes_model(preprocessor)
}

print(f"✅ {len(competitors)} modèles créés:")
for name in competitors.keys():
    print(f"   - {name}")

## 4️⃣ Réglage des Hyperparamètres (Optionnel)

⚠️ Cette étape peut prendre du temps. Mettez `enable_tuning = True` pour activer.

In [ ]:
def get_param_grids():
    """Grilles de paramètres pour le tuning"""
    param_grids = {
        'Logistic Regression': {
            'classifier__C': [0.1, 1, 10],
            'classifier__max_iter': [1000, 5000]
        },
        'Decision Tree': {
            'classifier__max_depth': [5, 10, 15],
            'classifier__min_samples_split': [2, 5],
            'classifier__min_samples_leaf': [1, 2]
        },
        'Random Forest': {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [10, 15, 20],
            'classifier__min_samples_split': [2, 5]
        },
        'Naive Bayes': {
            'classifier__var_smoothing': [1e-9, 1e-8, 1e-7]
        },
        'Support Vector Machine': {
            'classifier__estimator__C': [0.1, 1, 10],
            'classifier__estimator__max_iter': [1000, 5000]
        },
        'Perceptron': {
            'classifier__estimator__alpha': [0.001, 0.01, 0.1],
            'classifier__estimator__max_iter': [1000, 5000]
        }
    }
    return param_grids

def tune_model_with_validation(model_pipeline, X_train, y_train, X_val, y_val, model_name, scoring='f1'):
    """
    Optimiser un modèle avec GridSearchCV en utilisant le set de validation.
    Plus rigoureux que la cross-validation car le test set reste complètement séparé.
    """
    param_grids = get_param_grids()
    
    if model_name not in param_grids:
        print(f"[Warning] Pas de grille définie pour {model_name}")
        return model_pipeline, {}, None
    
    param_grid = param_grids[model_name]
    
    print(f"\n[Tuning] Optimisation: {model_name}")
    print(f"  Combinaisons à tester: {np.prod([len(v) for v in param_grid.values()])}")
    
    # GridSearch avec validation manuelle sur le set de validation
    from sklearn.model_selection import PredefinedSplit
    
    # Créer un split qui utilise train pour entraîner et val pour valider
    # -1 = train, 0 = validation
    split_index = [-1] * len(X_train) + [0] * len(X_val)
    
    # Concaténer train et val temporairement pour GridSearchCV
    X_combined = pd.concat([X_train, X_val])
    y_combined = pd.concat([y_train, y_val])
    
    ps = PredefinedSplit(test_fold=split_index)
    
    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=ps,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_combined, y_combined)
    
    print(f"  ✅ Meilleurs params: {grid_search.best_params_}")
    print(f"  ✅ Score validation ({scoring}): {grid_search.best_score_:.4f}")
    
    # Réentraîner sur train uniquement avec les meilleurs paramètres
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    return best_model, grid_search.best_params_, grid_search.best_score_

def tune_all_models(competitors, X_train, y_train, X_val, y_val, scoring='f1'):
    """Optimiser tous les modèles avec le set de validation"""
    tuned_competitors = {}
    tuning_results = {}
    
    print("\n" + "="*60)
    print("🔧 Début du réglage des hyperparamètres (avec set de validation)")
    print("="*60)
    
    for model_name, pipeline in competitors.items():
        best_pipeline, best_params, best_score = tune_model_with_validation(
            pipeline, X_train, y_train, X_val, y_val, model_name, scoring=scoring
        )
        tuned_competitors[model_name] = best_pipeline
        tuning_results[model_name] = (best_params, best_score)
    
    print("\n" + "="*60)
    print("✅ Réglage terminé")
    print("="*60)
    
    # Afficher le résumé
    summary_data = []
    for model_name, (best_params, best_score) in tuning_results.items():
        summary_data.append({
            'Model': model_name,
            'Validation Score (F1)': f"{best_score:.4f}" if best_score else "N/A",
            'Num Params Tuned': len(best_params) if best_params else 0
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n📊 Résumé du tuning (sur set de validation):")
    display(summary_df)
    
    return tuned_competitors, tuning_results

print("✅ Fonctions de tuning définies (avec split validation)")

In [ ]:
# Configuration du tuning
enable_tuning = True  # Mettre à True pour activer le tuning

if enable_tuning:
    print("🔧 Tuning activé - Optimisation sur le set de validation")
    competitors, tuning_results = tune_all_models(competitors, X_train, y_train, X_val, y_val, scoring='f1')
else:
    print("⏭️ Tuning désactivé. Utilisation des hyperparamètres par défaut.")
    print("💡 Conseil: Activez le tuning pour de meilleures performances !")

## 4️⃣.1 Advanced Tuning - Random Forest 🌲

Cette section effectue un tuning approfondi du Random Forest avec une grille de paramètres étendue pour obtenir les meilleures performances possibles.

In [ ]:
def get_advanced_rf_param_grid():
    """
    Grille de paramètres étendue pour Random Forest.
    Teste ~1080 combinaisons pour trouver la configuration optimale.
    """
    param_grid = {
        # Nombre d'arbres dans la forêt
        'classifier__n_estimators': [100, 200, 300, 500],
        
        # Profondeur maximale des arbres
        'classifier__max_depth': [10, 15, 20, 25, 30, None],
        
        # Nombre minimum d'échantillons pour split
        'classifier__min_samples_split': [2, 5, 10, 15],
        
        # Nombre minimum d'échantillons par feuille
        'classifier__min_samples_leaf': [1, 2, 4, 6],
        
        # Nombre de features à considérer pour le meilleur split
        'classifier__max_features': ['sqrt', 'log2', None],
        
        # Bootstrap (avec ou sans remplacement)
        'classifier__bootstrap': [True, False],
        
        # Critère de qualité du split
        'classifier__criterion': ['gini', 'entropy']
    }
    
    # Calculer le nombre total de combinaisons
    import numpy as np
    total_combinations = np.prod([len(v) for v in param_grid.values()])
    
    print(f"📊 Grille de paramètres pour Random Forest:")
    print(f"   - n_estimators: {param_grid['classifier__n_estimators']}")
    print(f"   - max_depth: {param_grid['classifier__max_depth']}")
    print(f"   - min_samples_split: {param_grid['classifier__min_samples_split']}")
    print(f"   - min_samples_leaf: {param_grid['classifier__min_samples_leaf']}")
    print(f"   - max_features: {param_grid['classifier__max_features']}")
    print(f"   - bootstrap: {param_grid['classifier__bootstrap']}")
    print(f"   - criterion: {param_grid['classifier__criterion']}")
    print(f"\n🔢 Total de combinaisons à tester: {total_combinations}")
    
    return param_grid

print("✅ Grille avancée Random Forest définie")

In [ ]:
def advanced_tune_random_forest(X_train, y_train, X_val, y_val, preprocessor, scoring='f1', n_jobs=-1):
    """
    Tuning avancé du Random Forest avec RandomizedSearchCV pour plus d'efficacité.
    
    Args:
        X_train, y_train: Données d'entraînement
        X_val, y_val: Données de validation
        preprocessor: Pipeline de prétraitement
        scoring: Métrique d'optimisation
        n_jobs: Nombre de processus parallèles (-1 = tous)
    
    Returns:
        best_model: Meilleur modèle entraîné
        best_params: Meilleurs hyperparamètres
        best_score: Meilleur score sur validation
        cv_results: Résultats détaillés du tuning
    """
    from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
    
    print("\n" + "="*70)
    print("🔬 ADVANCED TUNING - RANDOM FOREST")
    print("="*70)
    
    # Créer le modèle de base
    base_rf = RandomForestClassifier(random_state=42)
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', base_rf)
    ])
    
    # Obtenir la grille de paramètres
    param_grid = get_advanced_rf_param_grid()
    
    # Créer le split pour validation
    split_index = [-1] * len(X_train) + [0] * len(X_val)
    X_combined = pd.concat([X_train, X_val])
    y_combined = pd.concat([y_train, y_val])
    ps = PredefinedSplit(test_fold=split_index)
    
    # Utiliser RandomizedSearchCV pour plus d'efficacité
    # (teste un sous-ensemble aléatoire des combinaisons)
    print(f"\n🎲 Utilisation de RandomizedSearchCV (300 itérations max)")
    print(f"⏱️ Ceci peut prendre plusieurs minutes...")
    
    random_search = RandomizedSearchCV(
        rf_pipeline,
        param_distributions=param_grid,
        n_iter=300,  # Nombre d'itérations (combinaisons à tester)
        cv=ps,
        scoring=scoring,
        n_jobs=n_jobs,
        verbose=2,
        random_state=42,
        return_train_score=True
    )
    
    # Lancer le tuning
    random_search.fit(X_combined, y_combined)
    
    print(f"\n{'='*70}")
    print("✅ TUNING TERMINÉ")
    print(f"{'='*70}")
    
    # Afficher les résultats
    print(f"\n🏆 Meilleurs paramètres:")
    for param, value in random_search.best_params_.items():
        print(f"   - {param.replace('classifier__', '')}: {value}")
    
    print(f"\n📊 Scores:")
    print(f"   - Score validation ({scoring}): {random_search.best_score_:.4f}")
    
    # Réentraîner sur train uniquement avec les meilleurs paramètres
    best_model = random_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    # Évaluer sur validation
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    
    print(f"\n✅ Performance après réentraînement sur train:")
    print(f"   - Validation F1-Score: {val_f1:.4f}")
    print(f"   - Validation Accuracy: {val_acc:.4f}")
    
    # Créer un DataFrame avec les meilleurs résultats
    results_df = pd.DataFrame(random_search.cv_results_)
    results_df = results_df.sort_values('rank_test_score')
    top_10 = results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)
    
    print(f"\n🔝 Top 10 des configurations:")
    display(top_10)
    
    return best_model, random_search.best_params_, random_search.best_score_, random_search.cv_results_

print("✅ Fonction de tuning avancé définie")

In [ ]:
# Lancer le tuning avancé du Random Forest
enable_advanced_rf_tuning = True  # Mettre à True pour activer

if enable_advanced_rf_tuning:
    print("🚀 Lancement du tuning avancé Random Forest...")
    
    # Lancer le tuning
    best_rf_model, best_rf_params, best_rf_score, rf_cv_results = advanced_tune_random_forest(
        X_train, y_train, 
        X_val, y_val,
        preprocessor,
        scoring='f1',
        n_jobs=-1  # Utiliser tous les coeurs disponibles
    )
    
    # Remplacer le Random Forest dans le dictionnaire des compétiteurs
    competitors['Random Forest (Advanced)'] = best_rf_model
    
    print(f"\n✅ Random Forest optimisé ajouté aux compétiteurs!")
    
else:
    print("⏭️ Tuning avancé Random Forest désactivé.")
    print("💡 Activez-le pour obtenir les meilleures performances possibles !")
    print("⚠️ Attention: Cela peut prendre 10-30 minutes selon votre machine.")

### 📊 Analyse des résultats du tuning avancé

Cette section permet de visualiser l'impact des différents paramètres sur les performances.

In [ ]:
# Visualiser les résultats du tuning (si activé)
if enable_advanced_rf_tuning and 'rf_cv_results' in locals():
    print("📈 Analyse des résultats du tuning avancé\n")
    
    # Créer un DataFrame avec les résultats
    results_analysis = pd.DataFrame(rf_cv_results)
    
    # 1. Distribution des scores
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Score distribution
    axes[0, 0].hist(results_analysis['mean_test_score'], bins=30, color='skyblue', edgecolor='black')
    axes[0, 0].axvline(best_rf_score, color='red', linestyle='--', linewidth=2, label=f'Meilleur: {best_rf_score:.4f}')
    axes[0, 0].set_xlabel('Score F1 (Validation)')
    axes[0, 0].set_ylabel('Fréquence')
    axes[0, 0].set_title('Distribution des Scores de Validation')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Impact du nombre d'arbres
    if 'param_classifier__n_estimators' in results_analysis.columns:
        n_est_data = results_analysis.groupby('param_classifier__n_estimators')['mean_test_score'].agg(['mean', 'std'])
        axes[0, 1].errorbar(n_est_data.index, n_est_data['mean'], yerr=n_est_data['std'], 
                           marker='o', capsize=5, linewidth=2, markersize=8)
        axes[0, 1].set_xlabel('Nombre d\'arbres (n_estimators)')
        axes[0, 1].set_ylabel('Score F1 moyen')
        axes[0, 1].set_title('Impact du nombre d\'arbres')
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Impact de la profondeur
    if 'param_classifier__max_depth' in results_analysis.columns:
        depth_data = results_analysis.groupby('param_classifier__max_depth')['mean_test_score'].agg(['mean', 'std'])
        axes[1, 0].errorbar(range(len(depth_data)), depth_data['mean'], yerr=depth_data['std'],
                           marker='s', capsize=5, linewidth=2, markersize=8, color='green')
        axes[1, 0].set_xticks(range(len(depth_data)))
        axes[1, 0].set_xticklabels([str(x) if x is not None else 'None' for x in depth_data.index], rotation=45)
        axes[1, 0].set_xlabel('Profondeur max (max_depth)')
        axes[1, 0].set_ylabel('Score F1 moyen')
        axes[1, 0].set_title('Impact de la profondeur des arbres')
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Top 20 configurations
    top_20 = results_analysis.nsmallest(20, 'rank_test_score')
    axes[1, 1].barh(range(20), top_20['mean_test_score'], color='coral')
    axes[1, 1].set_xlabel('Score F1')
    axes[1, 1].set_ylabel('Rang de la configuration')
    axes[1, 1].set_title('Top 20 des configurations')
    axes[1, 1].invert_yaxis()
    axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Afficher les statistiques
    print(f"\n📊 Statistiques du tuning:")
    print(f"   - Meilleur score: {results_analysis['mean_test_score'].max():.4f}")
    print(f"   - Score moyen: {results_analysis['mean_test_score'].mean():.4f}")
    print(f"   - Score médian: {results_analysis['mean_test_score'].median():.4f}")
    print(f"   - Écart-type: {results_analysis['mean_test_score'].std():.4f}")
    
else:
    print("ℹ️ Activez enable_advanced_rf_tuning pour voir l'analyse des résultats")

## 5️⃣ Entraînement et Évaluation des Modèles

⚠️ **Important**: L'évaluation se fait sur le **set de test** (15%) qui n'a jamais été vu pendant l'entraînement ou le tuning !

In [ ]:
def plot_confusion_matrix(model_name, cm):
    """Dessiner la matrice de confusion"""
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'Matrice de Confusion - {model_name}')
    plt.xlabel('Prédiction')
    plt.ylabel('Réalité')
    plt.tight_layout()
    plt.show()

def evaluate_model_detailed(model, X_test, y_test, model_name):
    """Évaluation détaillée d'un modèle"""
    print(f"\n{'='*60}")
    print(f"Évaluation: {model_name}")
    print(f"{'='*60}")
    
    # Prédictions
    y_pred = model.predict(X_test)
    
    # Rapport de classification
    print("\n📊 Rapport de classification:")
    print(classification_report(y_test, y_pred))
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n🔢 Matrice de confusion:")
    print(f"TP (VP): {cm[1][1]} | FP (FP): {cm[0][1]}")
    print(f"FN (FN): {cm[1][0]} | TN (VN): {cm[0][0]}")
    
    plot_confusion_matrix(model_name, cm)
    
    # Métriques
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    # Calculer AUC
    auc = "N/A"
    try:
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test)
            if proba.ndim == 2:
                auc = roc_auc_score(y_test, proba[:, 1])
            else:
                auc = roc_auc_score(y_test, proba)
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X_test)
            if getattr(scores, "ndim", 1) == 2:
                auc = roc_auc_score(y_test, scores[:, 1])
            else:
                auc = roc_auc_score(y_test, scores)
    except Exception:
        pass
    
    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1-Score": f1,
        "AUC": auc
    }

print("✅ Fonction d'évaluation définie")

In [ ]:
# Entraîner et évaluer tous les modèles sur le TEST SET
print("\n" + "="*60)
print("🚀 ENTRAÎNEMENT ET ÉVALUATION")
print("="*60)
print("📌 Entraînement: Train set (70%)")
print("📌 Évaluation: Test set (15%) - Jamais vu !")
print("="*60)

trained_models = {}
results_summary = []

for name, pipeline in competitors.items():
    print(f"\n🔄 Entraînement: {name}...")
    
    # Entraîner sur le train set uniquement
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    
    # Évaluation détaillée sur le TEST SET (complètement séparé)
    metrics = evaluate_model_detailed(pipeline, X_test, y_test, name)
    results_summary.append(metrics)

print("\n" + "="*60)
print("✅ ENTRAÎNEMENT ET ÉVALUATION TERMINÉS")
print("="*60)

## 6️⃣ Comparaison des Modèles

In [ ]:
# Table de comparaison
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values(by="F1-Score", ascending=False)

print("\n" + "#"*60)
print("🏆 COMPARAISON DES MODÈLES")
print("#"*60 + "\n")

display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"\n🥇 Meilleur modèle: {best_model_name}")

In [ ]:
# Visualisation des performances
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
axes[0].barh(results_df['Model'], results_df['Accuracy'], color='skyblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Comparaison - Accuracy')
axes[0].set_xlim([0, 1])

# F1-Score
axes[1].barh(results_df['Model'], results_df['F1-Score'], color='lightgreen')
axes[1].set_xlabel('F1-Score')
axes[1].set_title('Comparaison - F1-Score')
axes[1].set_xlim([0, 1])

# AUC
auc_values = [v if isinstance(v, (int, float)) else 0 for v in results_df['AUC']]
axes[2].barh(results_df['Model'], auc_values, color='salmon')
axes[2].set_xlabel('AUC')
axes[2].set_title('Comparaison - AUC')
axes[2].set_xlim([0, 1])

plt.tight_layout()
plt.show()

## 7️⃣ Courbes ROC

In [ ]:
# Courbes ROC pour tous les modèles
plt.figure(figsize=(10, 7))

for name, pipeline in trained_models.items():
    try:
        if hasattr(pipeline, 'predict_proba'):
            proba = pipeline.predict_proba(X_test)
            scores = proba[:, 1] if proba.ndim == 2 else proba
        elif hasattr(pipeline, 'decision_function'):
            scores = pipeline.decision_function(X_test)
            if getattr(scores, 'ndim', 1) == 2:
                scores = scores[:, 1]
        else:
            continue

        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_score = roc_auc_score(y_test, scores)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})', linewidth=2)
    except Exception:
        continue

plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire', linewidth=1)
plt.title('Courbes ROC - Tous les Modèles', fontsize=14)
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8️⃣ Courbes Précision-Rappel

In [ ]:
# Courbes Précision-Rappel pour tous les modèles
plt.figure(figsize=(10, 7))

for name, pipeline in trained_models.items():
    try:
        if hasattr(pipeline, 'predict_proba'):
            proba = pipeline.predict_proba(X_test)
            scores = proba[:, 1] if proba.ndim == 2 else proba
        elif hasattr(pipeline, 'decision_function'):
            scores = pipeline.decision_function(X_test)
            scores = scores[:, 1] if getattr(scores, 'ndim', 1) == 2 else scores
        else:
            continue
        
        precision, recall, _ = precision_recall_curve(y_test, scores)
        ap = average_precision_score(y_test, scores)
        plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})", linewidth=2)
    except Exception:
        continue

plt.title('Courbes Précision-Rappel - Tous les Modèles', fontsize=14)
plt.xlabel('Rappel', fontsize=12)
plt.ylabel('Précision', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9️⃣ Validation Croisée (5-Fold) - Optionnel

Cette validation se fait sur le **train set uniquement** pour évaluer la stabilité des modèles.

In [ ]:
def cross_validate_models(competitors, X, y, n_splits=5):
    """Validation croisée stratifiée pour tous les modèles"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_summary = []
    
    print("\n" + "="*60)
    print(f"🔄 Validation Croisée {n_splits}-Fold")
    print("="*60 + "\n")
    
    for name, pipeline in competitors.items():
        print(f"Validation: {name}...")
        auc_scores, f1_scores, acc_scores, ap_scores = [], [], [], []
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
            
            pipeline.fit(X_tr, y_tr)
            y_pred = pipeline.predict(X_te)
            
            acc_scores.append(accuracy_score(y_te, y_pred))
            f1_scores.append(f1_score(y_te, y_pred))
            
            # Calculer AUC et AP
            scores = None
            if hasattr(pipeline, 'predict_proba'):
                proba = pipeline.predict_proba(X_te)
                scores = proba[:, 1] if proba.ndim == 2 else proba
            elif hasattr(pipeline, 'decision_function'):
                scores = pipeline.decision_function(X_te)
                scores = scores[:, 1] if getattr(scores, 'ndim', 1) == 2 else scores
            
            if scores is not None:
                auc_scores.append(roc_auc_score(y_te, scores))
                ap_scores.append(average_precision_score(y_te, scores))
        
        cv_summary.append({
            'Model': name,
            'Acc(mean)': np.mean(acc_scores) if acc_scores else None,
            'F1(mean)': np.mean(f1_scores) if f1_scores else None,
            'AUC(mean)': np.mean(auc_scores) if auc_scores else None,
            'AP(mean)': np.mean(ap_scores) if ap_scores else None,
        })
    
    df = pd.DataFrame(cv_summary)
    print("\n" + "="*60)
    print("📊 Résultats de la Validation Croisée (moyennes)")
    print("="*60 + "\n")
    display(df)
    
    return df

# Exécuter la validation croisée
cv_results = cross_validate_models(competitors, X_train, y_train, n_splits=5)

## 🔟 Sauvegarde du Meilleur Modèle

Cette section permet de sauvegarder le modèle ayant obtenu les meilleures performances pour une utilisation future en production.

In [ ]:
import joblib
import json
from datetime import datetime

# Identifier le meilleur modèle
print("🔍 Identification du meilleur modèle...\n")

# Récupérer le modèle avec le meilleur F1-Score
best_model_name = results_df.iloc[0]["Model"]
best_f1_score = results_df.iloc[0]["F1-Score"]
best_accuracy = results_df.iloc[0]["Accuracy"]
best_auc = results_df.iloc[0]["AUC"]

# Récupérer le pipeline du modèle
best_model = trained_models[best_model_name]

print(f"🏆 Meilleur modèle identifié:")
print(f"   - Nom: {best_model_name}")
print(f"   - F1-Score: {best_f1_score:.4f}")
print(f"   - Accuracy: {best_accuracy:.4f}")
print(f"   - AUC: {best_auc if isinstance(best_auc, str) else f'{best_auc:.4f}'}")

# Créer un dossier pour les modèles
models_dir = "saved_models"
import os
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"\n📁 Dossier '{models_dir}' créé")

# Timestamp pour le nom du fichier
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"{models_dir}/best_model_{timestamp}.joblib"
info_filename = f"{models_dir}/best_model_{timestamp}_info.json"

print(f"\n💾 Sauvegarde du modèle...")

In [ ]:
# Sauvegarder le modèle avec joblib
joblib.dump(best_model, model_filename)
print(f"✅ Modèle sauvegardé: {model_filename}")

# Préparer les informations du modèle
model_info = {
    "model_name": best_model_name,
    "timestamp": timestamp,
    "date_training": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "performance": {
        "f1_score": float(best_f1_score),
        "accuracy": float(best_accuracy),
        "auc": float(best_auc) if isinstance(best_auc, (int, float)) else best_auc
    },
    "dataset": {
        "train_size": len(X_train),
        "validation_size": len(X_val),
        "test_size": len(X_test),
        "split_ratio": "70/15/15"
    },
    "hyperparameters": {}
}

# Ajouter les hyperparamètres si disponibles
if enable_tuning and best_model_name in tuning_results:
    best_params, _ = tuning_results[best_model_name]
    model_info["hyperparameters"] = best_params
elif enable_advanced_rf_tuning and best_model_name == "Random Forest (Advanced)":
    model_info["hyperparameters"] = best_rf_params

# Sauvegarder les métadonnées en JSON
with open(info_filename, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=4, ensure_ascii=False)

print(f"✅ Métadonnées sauvegardées: {info_filename}")

print(f"\n📊 Résumé de la sauvegarde:")
print(f"   - Fichier modèle: {model_filename}")
print(f"   - Fichier métadonnées: {info_filename}")
print(f"   - Taille du modèle: {os.path.getsize(model_filename) / (1024*1024):.2f} MB")

# Afficher les métadonnées
print(f"\n📋 Métadonnées du modèle:")
display(pd.DataFrame([model_info["performance"]]).T.rename(columns={0: "Valeur"}))

### 📥 Test de Rechargement du Modèle

Vérifions que le modèle peut être rechargé correctement.

In [ ]:
# Recharger le modèle et les métadonnées
print("🔄 Test de rechargement du modèle...\n")

# Charger le modèle
loaded_model = joblib.load(model_filename)
print(f"✅ Modèle rechargé depuis: {model_filename}")

# Charger les métadonnées
with open(info_filename, 'r', encoding='utf-8') as f:
    loaded_info = json.load(f)

print(f"✅ Métadonnées rechargées depuis: {info_filename}")

# Vérifier que le modèle fonctionne
print(f"\n🧪 Test de prédiction sur quelques exemples du test set...")

# Prendre 5 exemples du test set
sample_indices = list(range(5))
X_sample = X_test.iloc[sample_indices]
y_sample = y_test.iloc[sample_indices]

# Faire des prédictions
predictions = loaded_model.predict(X_sample)
probabilities = loaded_model.predict_proba(X_sample)[:, 1]

# Afficher les résultats
print(f"\n📊 Résultats du test:")
test_results = pd.DataFrame({
    'Index': sample_indices,
    'Réalité': y_sample.values,
    'Prédiction': predictions,
    'Probabilité Départ': [f"{p:.2%}" for p in probabilities],
    'Statut': ['✅ Correct' if predictions[i] == y_sample.iloc[i] else '❌ Incorrect' 
               for i in range(len(predictions))]
})

display(test_results)

# Calculer l'exactitude sur l'échantillon
correct_predictions = (predictions == y_sample.values).sum()
print(f"\n✅ Précision sur l'échantillon: {correct_predictions}/{len(predictions)} ({correct_predictions/len(predictions):.1%})")

print(f"\n🎉 Le modèle a été sauvegardé et rechargé avec succès !")

## 🔟 Bibliographie



#### Introduction Méthodologique

Cette bibliographie annotée présente les références académiques et techniques qui ont orienté notre travail dans le cadre du projet de prédiction de l'attrition des employés dans une entreprise confrontée à un taux de rotation élevé de 15% annuel.

Les sources sélectionnées ont été organisées selon six axes thématiques correspondant aux différentes dimensions du projet : les fondements méthodologiques du Machine Learning et du Data Mining, les aspects techniques de mise en œuvre des algorithmes de classification binaire, les techniques de gestion du déséquilibre de classes, les méthodes de validation et d'évaluation des modèles prédictifs, les aspects éthiques et réglementaires, ainsi que les technologies et outils de développement.

Chaque source est accompagnée d'une annotation justifiant son intégration dans notre démarche, explicitant son apport théorique ou pratique, et précisant son impact sur nos choix méthodologiques, techniques ou éthiques. L'ensemble de ces références a permis de construire une approche rigoureuse, éthiquement responsable et techniquement robuste pour répondre aux objectifs du projet de prédiction de l'attrition des employés.

---

#### Axe 1 : Fondements Méthodologiques et Cadre Théorique

##### 1.1 Méthodologie CRISP-DM

**Wirth, R., & Hipp, J. (2000).** CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29-39.

Cet article fondateur présente le modèle CRISP-DM (Cross-Industry Standard Process for Data Mining), qui structure notre approche méthodologique du projet. Le modèle définit six phases itératives : compréhension métier, compréhension des données, préparation des données, modélisation, évaluation et déploiement. Dans notre contexte, cette méthodologie nous a guidés pour structurer rigoureusement notre workflow, de l'analyse du problème d'attrition (taux de rotation de 15% annuel) jusqu'à la préparation de cinq datasets distincts (general_data, manager_survey, employee_survey, in_time, out_time) et la sélection des algorithmes de classification binaire (Logistic Regression, Decision Tree, Random Forest, SVM, Naive Bayes, Perceptron). L'approche itérative préconisée nous a permis d'améliorer continuellement nos modèles en revenant sur les phases antérieures lorsque les résultats d'évaluation ne se justifiaient pas. Cette référence constitue le socle méthodologique justifiant notre démarche structurée et reproductible.

##### 1.2 Classification Binaire et Métriques d'Évaluation

**Sokolova, M., & Lapalme, G. (2009).** A systematic analysis of performance measures for classification tasks. *Information Processing & Management*, 45(4), 427-437. https://doi.org/10.1016/j.ipm.2009.03.002

Cette analyse systématique des mesures de performance pour les tâches de classification a directement influencé nos choix de métriques d'évaluation dans le projet. Les auteurs comparent exhaustivement les métriques standards (précision, rappel, F1-score, spécificité) et expliquent leur pertinence selon le contexte métier. Pour notre problème de prédiction d'attrition avec une distribution déséquilibrée (85% non-départ / 15% départ), cette source nous a aidés à privilégier le F1-score et l'AUC-ROC plutôt que la simple accuracy, qui serait trompeuse. L'article détaille également l'importance de la matrice de confusion pour interpréter les faux positifs et faux négatifs dans un contexte métier où prédire à tort un départ (faux positif) ou manquer un départ réel (faux négatif) ont des impacts différents sur l'entreprise. Ces considérations ont guidé notre stratégie d'évaluation et d'amélioration continue des modèles, notamment dans le choix des seuils de décision optimaux.

##### 1.3 Prétraitement et Pipeline Scikit-learn

**Buitinck, L., Louppe, G., Blondel, M., Pedregosa, F., Mueller, A., Grisel, O., ... & Varoquaux, G. (2013).** API design for machine learning software: experiences from the scikit-learn project. *arXiv preprint arXiv:1309.0238*.

Cette référence documente l'architecture et les principes de conception de scikit-learn, la bibliothèque Python que nous avons utilisée pour l'ensemble de notre pipeline ML. Les auteurs expliquent le concept de Pipeline qui permet de chaîner les étapes de prétraitement (imputation des valeurs manquantes via SimpleImputer, encodage one-hot des variables catégorielles via OneHotEncoder, standardisation des variables numériques via StandardScaler) et de modélisation de manière cohérente et reproductible. Dans notre projet, nous avons implémenté un ColumnTransformer pour appliquer différents prétraitements aux variables numériques et catégorielles, puis intégré ce transformeur dans des Pipelines pour chaque modèle. Cette approche garantit que les mêmes transformations sont appliquées de manière identique sur les données d'entraînement, de validation et de test, évitant ainsi les fuites de données (data leakage). L'article justifie également notre utilisation de la validation croisée stratifiée (StratifiedKFold) pour évaluer robustement les performances, particulièrement crucial avec notre déséquilibre de classes.

---

#### Axe 2 : Algorithmes de Classification et Comparaison de Modèles

##### 2.1 Régression Logistique

**Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013).** *Applied Logistic Regression* (3rd ed.). Wiley.

Ce manuel de référence sur la régression logistique fournit les fondements théoriques et pratiques de l'un de nos modèles principaux. La régression logistique modélise la probabilité d'attrition en fonction des variables prédictives via une fonction logistique (sigmoïde), permettant d'obtenir des prédictions probabilistes interprétables. Les auteurs détaillent les méthodes d'estimation des coefficients par maximum de vraisemblance, l'interprétation des odds ratios pour quantifier l'impact de chaque variable, et les diagnostics de qualité d'ajustement. Dans notre projet, nous avons configuré LogisticRegression avec un paramètre de régularisation C (inverse de la force de régularisation) et max_iter=1000 pour assurer la convergence. Cette source a guidé notre interprétation des coefficients du modèle pour identifier les facteurs RH les plus influents sur l'attrition (distance domicile-travail, satisfaction professionnelle, équilibre vie professionnelle/personnelle), fournissant ainsi des insights actionnables pour les décideurs RH au-delà de la simple prédiction.

##### 2.2 Arbres de Décision et Random Forest

**Breiman, L. (2001).** Random Forests. *Machine Learning*, 45(1), 5-32.

Cet article fondateur de Leo Breiman introduit l'algorithme Random Forest, une méthode d'ensemble basée sur l'agrégation de multiples arbres de décision. Les Random Forests construisent des arbres sur des sous-ensembles bootstrapés des données et avec des sous-ensembles aléatoires de variables à chaque nœud, réduisant ainsi la variance et le surapprentissage inhérents aux arbres individuels. Dans notre projet, Random Forest s'est révélé être l'un des modèles les plus performants grâce à sa capacité à capturer des interactions non-linéaires complexes entre variables RH. Nous avons configuré le modèle avec n_estimators (100-200 arbres), max_depth (10-20) et min_samples_split pour contrôler la complexité. L'article explique également l'importance de l'Out-of-Bag (OOB) error et du feature importance, que nous avons utilisés pour identifier les variables RH les plus prédictives de l'attrition. Cette approche nous a permis de combiner haute performance prédictive et interprétabilité relative, essentielle pour la confiance des utilisateurs métier.

##### 2.3 Support Vector Machines (SVM)

**Cortes, C., & Vapnik, V. (1995).** Support-vector networks. *Machine Learning*, 20(3), 273-297.

Cette publication originale sur les Support Vector Machines présente les fondements théoriques de l'algorithme SVM, qui recherche l'hyperplan de séparation optimal maximisant la marge entre les classes. Bien que développé initialement pour la classification linéaire, l'article introduit le concept de kernel trick permettant d'étendre SVM aux problèmes non-linéaires. Dans notre projet, nous avons utilisé LinearSVC (Support Vector Classification avec noyau linéaire) de scikit-learn, particulièrement adapté à notre dataset de dimension moyenne après one-hot encoding. La formulation par SVM offre une robustesse théorique grâce à la maximisation de la marge, réduisant le risque de surapprentissage même avec des données bruitées. Nous avons ajusté le paramètre de régularisation C pour contrôler le compromis biais-variance. Cette source justifie notre inclusion de SVM dans la comparaison de modèles, apportant une perspective géométrique complémentaire aux approches probabilistes (Logistic Regression, Naive Bayes) et par arbres (Decision Tree, Random Forest).

##### 2.4 Naive Bayes

**Rish, I. (2001).** An empirical study of the naive Bayes classifier. *IJCAI 2001 Workshop on Empirical Methods in Artificial Intelligence*, 3(22), 41-46.

Cet article empirique évalue les performances du classificateur Naive Bayes, qui repose sur le théorème de Bayes avec l'hypothèse d'indépendance conditionnelle des variables explicatives. Malgré cette hypothèse souvent violée en pratique, l'article montre que Naive Bayes peut atteindre des performances compétitives, particulièrement sur des données de haute dimension. Dans notre projet RH, nous avons implémenté GaussianNB qui suppose une distribution gaussienne des variables continues après standardisation. L'avantage majeur de Naive Bayes réside dans sa simplicité, sa rapidité d'entraînement et sa capacité à fournir des probabilités calibrées naturellement. Cette source a justifié notre inclusion de Naive Bayes comme baseline rapide et interprétable, offrant un point de comparaison pour évaluer si des modèles plus complexes (Random Forest, SVM) apportent un gain de performance suffisant pour justifier leur complexité accrue.

##### 2.5 Perceptron et Modèles Linéaires

**Rosenblatt, F. (1958).** The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review*, 65(6), 386-408.

Article historique introduisant le Perceptron, l'un des premiers algorithmes d'apprentissage supervisé, précurseur des réseaux de neurones modernes. Le Perceptron est un classificateur linéaire qui apprend itérativement les poids en ajustant la frontière de décision à chaque erreur de classification. Dans notre projet, nous avons implémenté le Perceptron via CalibratedClassifierCV pour obtenir des probabilités calibrées (le Perceptron standard ne fournit que des prédictions binaires). Bien que théoriquement limité aux problèmes linéairement séparables, le Perceptron a servi de baseline algorithmique simple pour notre comparaison de modèles. Cette référence fondamentale éclaire les racines historiques du Machine Learning et contextualise notre approche comparative : en partant du Perceptron simple jusqu'aux ensembles complexes (Random Forest), nous avons pu quantifier le gain de performance apporté par la complexité algorithmique croissante, tout en gardant une perspective critique sur le compromis interprétabilité-performance.

---

#### Axe 3 : Gestion du Déséquilibre de Classes

##### 3.1 SMOTE (Synthetic Minority Over-sampling Technique)

**Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002).** SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321-357. https://doi.org/10.1613/jair.953

Article fondamental présentant SMOTE (Synthetic Minority Over-sampling Technique), une technique cruciale pour notre problème de déséquilibre de classes (85% non-attrition / 15% attrition). Contrairement au sur-échantillonnage aléatoire qui duplique simplement les exemples minoritaires, SMOTE génère des exemples synthétiques en interpolant entre les exemples minoritaires existants dans l'espace des features. Dans notre projet, cette approche nous a permis d'équilibrer notre dataset d'entraînement sans introduire d'overfitting lié à la duplication pure. L'article explique également comment SMOTE améliore la capacité du modèle à généraliser sur la classe minoritaire (les départs) en créant des variations réalistes dans l'espace des features. Nous avons considéré l'implémentation de SMOTE via la bibliothèque imbalanced-learn avec un ajustement des poids de classes (class_weight='balanced'), conformément aux recommandations des auteurs pour maximiser les performances en classification déséquilibrée.

##### 3.2 Class Weighting et Stratégies d'Échantillonnage

**He, H., & Garcia, E. A. (2009).** Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering*, 21(9), 1263-1284.

Cette revue exhaustive sur l'apprentissage à partir de données déséquilibrées présente un panorama complet des stratégies pour traiter ce problème récurrent en ML. Les auteurs comparent les approches au niveau des données (sur-échantillonnage, sous-échantillonnage, SMOTE) et au niveau algorithmique (ajustement des poids de classes, modification des fonctions de coût). Dans notre projet d'attrition RH, nous avons combiné plusieurs stratégies : (1) considération de SMOTE pour générer des exemples synthétiques de départs, (2) class_weight='balanced' dans nos modèles pour pénaliser davantage les erreurs sur la classe minoritaire, et (3) validation croisée stratifiée pour maintenir la proportion des classes dans chaque fold. L'article souligne l'importance de choisir les bonnes métriques d'évaluation (F1-score, AUC-ROC) plutôt que l'accuracy qui peut être trompeuse avec des classes déséquilibrées.

---

#### Axe 4 : Validation, Évaluation et Optimisation des Modèles

##### 4.1 Validation Croisée et Bootstrap

**Kohavi, R. (1995).** A study of cross-validation and bootstrap for accuracy estimation and model selection. *Proceedings of the 14th International Joint Conference on Artificial Intelligence*, 2, 1137-1143.

Cette étude comparative entre validation croisée et bootstrap pour l'estimation de la performance des modèles a guidé notre stratégie d'évaluation. L'auteur démontre que la validation croisée k-fold, particulièrement avec k=5 ou k=10, fournit des estimations plus robustes et moins biaisées de l'erreur de généralisation que l'estimation simple train/test. Dans notre projet, nous avons implémenté StratifiedKFold avec 5 folds pour évaluer chaque modèle, garantissant que chaque fold maintient la proportion 85/15 de non-attrition/attrition. Cette approche est cruciale avec notre dataset et déséquilibre de classes : elle maximise l'utilisation des données d'entraînement tout en fournissant une estimation fiable des performances. L'article justifie également notre calcul des intervalles de confiance sur les métriques (accuracy, precision, recall, F1-score, AUC-ROC) à partir des résultats des 5 folds.

##### 4.2 Optimisation des Hyperparamètres - GridSearchCV

**Bergstra, J., & Bengio, Y. (2012).** Random search for hyper-parameter optimization. *Journal of Machine Learning Research*, 13, 281-305.

Bien que cet article promeuve la recherche aléatoire (Random Search) pour l'optimisation des hyperparamètres, il offre une perspective critique sur GridSearchCV que nous avons utilisé dans notre projet. GridSearchCV explore exhaustivement une grille de combinaisons d'hyperparamètres via validation croisée, identifiant la configuration optimale pour chaque modèle. Dans notre implémentation, nous avons défini des grilles de paramètres pour chaque algorithme : par exemple, pour Random Forest, nous avons testé différentes combinaisons de n_estimators (100, 200), max_depth (10, 15, 20), et min_samples_split (2, 5). L'article souligne que GridSearchCV, bien que coûteux computationnellement, garantit l'exploration complète de l'espace de recherche défini, contrairement à Random Search. Le tuning des hyperparamètres s'est révélé crucial, améliorant significativement les performances de base de chaque modèle.

##### 4.3 Interprétabilité et SHAP Values

**Lundberg, S. M., & Lee, S. I. (2017).** A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, 30, 4765-4774.

Cet article présente SHAP (SHapley Additive exPlanations), une méthode unifiée pour interpréter les prédictions des modèles de Machine Learning basée sur la théorie des jeux coopératifs. SHAP calcule la contribution de chaque variable à une prédiction individuelle, offrant une interprétabilité post-hoc même pour des modèles complexes comme Random Forest. Dans notre contexte d'attrition RH, l'interprétabilité est essentielle : les décideurs RH doivent comprendre pourquoi un employé est prédit comme à risque de départ pour pouvoir agir (proposer une augmentation, améliorer l'équilibre vie pro/perso, réduire la distance de trajet). Nous avons considéré implémenter SHAP pour expliquer nos prédictions de Random Forest, notre modèle le plus performant. L'article justifie l'importance de ne pas sacrifier totalement l'interprétabilité pour quelques points de performance.

##### 4.4 Métriques Avancées - AUC-ROC et Courbes de Précision-Rappel

**Davis, J., & Goadrich, M. (2006).** The relationship between Precision-Recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233-240.

Cet article analyse la relation entre les courbes ROC (Receiver Operating Characteristic) et Precision-Recall, deux visualisations complémentaires des performances de classification. Les auteurs démontrent que pour les problèmes avec classes déséquilibrées, les courbes Precision-Recall sont souvent plus informatives que les courbes ROC qui peuvent être optimistes. Dans notre projet d'attrition (15% classe positive), nous avons calculé à la fois l'AUC-ROC et considéré l'AUC-PR (Area Under Precision-Recall curve) pour chaque modèle. L'article explique que l'AUC-ROC mesure la capacité du modèle à discriminer entre les classes à travers tous les seuils de décision possibles, tandis que l'AUC-PR se concentre sur la performance de la classe positive (départs), plus pertinent pour notre cas d'usage métier.

---

#### Axe 5 : Aspects Éthiques et Réglementaires des RH

##### 5.1 RGPD et Protection des Données RH

**Voigt, P., & Von dem Bussche, A. (2017).** *The EU General Data Protection Regulation (GDPR): A Practical Guide*. Springer.

Ce guide pratique sur le Règlement Général sur la Protection des Données (RGPD) européen aborde les implications légales du traitement des données personnelles, particulièrement pertinent pour notre projet d'attrition RH qui manipule des données sensibles d'employés (satisfaction, équilibre vie pro/perso, évaluations manager, données temporelles d'arrivée/départ). Les auteurs détaillent les principes de minimisation des données, de finalité spécifique, de transparence et de droit d'accès des individus. Dans notre implémentation, nous avons veillé à : (1) anonymiser les EmployeeID dans nos analyses et visualisations, (2) limiter les variables aux strictement nécessaires pour la prédiction d'attrition, (3) documenter clairement l'objectif métier (réduction du turnover, non surveillance abusive), et (4) considérer les droits des employés à comprendre et contester les prédictions.

##### 5.2 Biais Algorithmiques et Équité en ML

**Mehrabi, N., Morstatter, F., Saxena, N., Lerman, K., & Galstyan, A. (2021).** A survey on bias and fairness in machine learning. *ACM Computing Surveys*, 54(6), 1-35.

Cette revue exhaustive sur les biais et l'équité en Machine Learning identifie différents types de biais (biais de sélection, biais de mesure, biais d'agrégation) et propose des stratégies d'atténuation. Dans notre projet RH de prédiction d'attrition, les risques de biais sont multiples : le dataset pourrait refléter des pratiques RH historiquement discriminatoires, et le modèle pourrait apprendre et amplifier ces biais. Les auteurs distinguent les notions d'équité individuelle (traiter similairement des individus similaires) et d'équité de groupe (égalité des taux de prédiction positive entre groupes démographiques). Bien que notre dataset ne contienne pas explicitement de variables protégées (genre, âge, origine ethnique), nous avons été vigilants sur les proxies potentiels. Cette source a sensibilisé notre équipe à auditer nos modèles pour détecter et corriger d'éventuelles disparités de performance entre sous-groupes.

##### 5.3 Éthique de l'IA en Gestion des Ressources Humaines

**Tambe, P., Cappelli, P., & Yakubovich, V. (2019).** Artificial intelligence in human resources management: Challenges and a path forward. *California Management Review*, 61(4), 15-42.

Cet article examine les opportunités et défis éthiques de l'IA appliquée aux RH, dont notre cas d'usage de prédiction d'attrition est un exemple emblématique. Les auteurs mettent en garde contre les risques de surveillance excessive, de déshumanisation des décisions RH, et de prophéties auto-réalisatrices (si un employé est étiqueté "à risque de départ", cela pourrait influencer négativement son traitement et provoquer effectivement son départ). Dans notre projet, ces considérations nous ont amenés à : (1) positionner l'outil comme une aide à la décision, non un système décisionnel automatisé, (2) privilégier l'explicabilité pour permettre aux RH de comprendre et questionner les prédictions, et (3) recommander un usage proactif et positif (identifier les employés à risque pour améliorer leurs conditions, non pour les stigmatiser). Cette perspective a façonné notre positionnement du projet comme un outil de prévention et de bien-être au travail.

---

#### Axe 6 : Technologies et Outils de Développement

##### 6.1 Python et Écosystème Data Science

**VanderPlas, J. (2016).** *Python Data Science Handbook: Essential Tools for Working with Data*. O'Reilly Media.

Ce manuel de référence couvre l'écosystème Python pour la Data Science, incluant les bibliothèques centrales de notre projet : NumPy pour les calculs numériques, Pandas pour la manipulation de dataframes (chargement et fusion de nos 5 fichiers CSV, gestion des valeurs manquantes, encodage de la variable cible Attrition), Matplotlib et Seaborn pour les visualisations exploratoires (distributions, corrélations, courbes ROC), et scikit-learn pour le Machine Learning. L'auteur explique les bonnes pratiques de travail avec des données tabulaires, la vectorisation pour l'efficacité computationnelle, et les patterns courants de preprocessing. Notre projet s'appuie massivement sur cet écosystème : Pandas pour créer notre Dataset_clean.csv à partir de general_data, manager_survey, employee_survey, in_time et out_time fusionnés sur EmployeeID.

##### 6.2 Jupyter Notebooks pour la Reproductibilité

**Kluyver, T., Ragan-Kelley, B., Pérez, F., Granger, B., Bussonnier, M., Frederic, J., ... & Jupyter Development Team. (2016).** Jupyter Notebooks—a publishing format for reproducible computational workflows. *Positioning and Power in Academic Publishing: Players, Agents and Agendas*, 87-90.

Cet article décrit Jupyter Notebook, l'environnement interactif que nous avons utilisé pour développer notre projet (ML_Project_Complete.ipynb). Jupyter permet de combiner code Python, visualisations, équations LaTeX et texte narratif dans un document unique exécutable, favorisant la reproductibilité et la communication des analyses. Les auteurs soulignent l'importance de Jupyter pour la science ouverte et la collaboration : notre notebook documente chronologiquement notre démarche (prétraitement → EDA → définition des modèles → tuning → entraînement → évaluation → comparaison), permettant à quiconque de reproduire nos résultats ou d'adapter le workflow à un nouveau dataset RH. Cette approche notebook-driven a facilité notre développement itératif, nos discussions d'équipe, et produira une livraison transparente et auto-documentée du projet aux décideurs RH.

##### 6.3 Versionnement et Collaboration - Git

**Chacon, S., & Straub, B. (2014).** *Pro Git* (2nd ed.). Apress.

Bien que non spécifiquement cité dans notre code, ce manuel sur Git (système de contrôle de version distribué) sous-tend notre méthodologie de collaboration de projet. Git nous a permis de versionner notre notebook Jupyter, de gérer les branches pour tester différentes approches, et de fusionner les contributions de l'équipe. Les auteurs expliquent les workflows Git (feature branches, pull requests) adaptés aux projets Data Science où les notebooks peuvent générer des conflits de merge complexes. Dans notre contexte, nous avons adopté un workflow Git simplifié : commits réguliers après chaque étape majeure (preprocessing, EDA, modélisation, évaluation), tags pour marquer les versions stables du projet, et .gitignore pour exclure les fichiers volumineux. Cette discipline de versionnement garantit la traçabilité de nos décisions méthodologiques, la récupération en cas d'erreur, et facilite l'audit ou la reprise du projet par de futurs développeurs ou auditeurs RH.

---

#### Conclusion Méthodologique

Cette bibliographie annotée reflète la diversité des connaissances mobilisées dans notre projet de prédiction d'attrition RH : des fondements méthodologiques (CRISP-DM) aux algorithmes de classification (Logistic Regression, Random Forest, SVM, Naive Bayes, Perceptron), en passant par la gestion du déséquilibre de classes (SMOTE, class weighting), l'optimisation (GridSearchCV, validation croisée), l'interprétabilité (SHAP), les aspects éthiques (RGPD, biais, IA responsable), et l'infrastructure technologique (Python, Jupyter, Git).

Chaque référence a contribué concrètement à nos choix méthodologiques, techniques ou éthiques, formant une base académique solide pour justifier notre approche. La combinaison de ces sources théoriques et pratiques nous a permis de développer un système de prédiction d'attrition performant, robuste (validé par cross-validation stratifiée), équitable (attention aux biais algorithmiques), et éthiquement responsable (conformité RGPD, explicabilité, aide à la décision non automatisation).

Cette bibliographie constitue également un point de départ pour des travaux futurs : approfondissement de l'interprétabilité via SHAP, expérimentation d'algorithmes de gradient boosting (XGBoost, LightGBM), déploiement en production avec monitoring des dérives de données et des biais, et expansion à d'autres problématiques RH prédictives (promotion, performance, engagement).

---

*Bibliographie rédigée dans le cadre du projet de formation ingénieur en Data Science et Intelligence Artificielle, CESI Nancy, 18 Décembre 2025.*